In [3]:
import os
import json
import pandas as pd
from datetime import datetime

# Path to your extracted Cricsheet JSON files
DATA_DIR = "raw_data"

parsed_matches = []

print("Starting to parse Cricsheet files...")

# Loop through all files in the directory
for filename in os.listdir(DATA_DIR):
    if filename.endswith(".json") and filename != "README.txt":
        file_path = os.path.join(DATA_DIR, filename)
        
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except Exception as e:
                continue
                
            # Extract basic metadata
            info = data.get("info", {})
            
            # Filter for Men's ODIs
            if info.get("match_type") != "ODI" or info.get("gender") != "male":
                continue
                
            # Extract and check the date (handling both list and string formats)
            dates = info.get("dates", [])
            if not dates:
                continue
            match_date_str = dates[0] # Take the first day of the match
            match_date = datetime.strptime(match_date_str, "%Y-%m-%d")
            
            # FILTER: Keep only matches from 2023 to 2026
            if not (2022 <= match_date.year <= 2026):
                continue
                
            venue = info.get("venue", "Unknown Venue")
            city = info.get("city", venue) # Fallback to venue name if city missing
            
            # Parse the innings data
            innings_list = data.get("innings", [])
            
            for index, innings_data in enumerate(innings_list):
                innings_num = index + 1
                if innings_num > 2: 
                    break # Skip super overs or 3rd/4th innings anomalies
                    
                team_name = innings_data.get("team")
                overs_data = innings_data.get("overs", [])
                
                total_runs = 0
                total_balls = 0
                
                # Calculate total runs and actual balls faced
                for over in overs_data:
                    for delivery in over.get("deliveries", []):
                        runs_dict = delivery.get("runs", {})
                        total_runs += runs_dict.get("total", 0)
                        total_balls += 1
                
                overs_played = round(total_balls / 6, 1)
                
                parsed_matches.append({
                    "match_id": filename.replace(".json", ""),
                    "date": match_date_str,
                    "venue": venue,
                    "city": city,
                    "innings": innings_num,
                    "team": team_name,
                    "team_total_score": total_runs,
                    "overs_played": overs_played
                })

# Convert to DataFrame
df_cricket = pd.DataFrame(parsed_matches)
print(f"Successfully processed {len(df_cricket)} innings rows from 2022-2026!")
print(df_cricket.head(4))

# Save this checkpoint
df_cricket.to_csv("cricket_base_dataset.csv", index=False)

Starting to parse Cricsheet files...
Successfully processed 1148 innings rows from 2022-2026!
  match_id        date                    venue    city  innings     team  \
0  1276907  2022-07-12  Kennington Oval, London  London        1  England   
1  1276907  2022-07-12  Kennington Oval, London  London        2    India   
2  1276908  2022-07-14           Lord's, London  London        1  England   
3  1276908  2022-07-14           Lord's, London  London        2    India   

   team_total_score  overs_played  
0               110          25.8  
1               114          19.5  
2               246          50.5  
3               146          39.0  
